In [1]:

import os
# You can create model using this method also,
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI

In [3]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# Muldimodal model (multi capabilities, can return image)

In [ ]:
response = model.invoke("Create a picture of a cat")
print(response.content_blocks)
# [
#     {"type": "text", "text": "Here's a picture of a cat"},
#     {"type": "image", "base64": "...", "mime_type": "image/jpeg"},
# ]

# ^^^^ the return will look like this

content='Here is a prompt for a picture of a cat:\n\n"A beautiful, fluffy tabby cat, gracefully sitting on a soft, sun-drenched windowsill. The cat has distinct brown and cream tabby markings, a gentle, curious expression, and large, bright green eyes that gaze softly forward. Its long, delicate whiskers are prominent. The fur looks incredibly soft and luxurious. The background is a gently blurred, cozy indoor setting, with warm natural light filtering in, creating a peaceful and serene atmosphere. Photorealistic, high detail, shallow depth of field, warm lighting."' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []} id='run--3020f65b-39cf-4307-964b-f7f5fc65e94e-0' usage_metadata={'input_tokens': 7, 'output_tokens': 114, 'total_tokens': 1133}


# Reasoning

Many models are capable of performing multi-step reasoning to arrive at a conclusion. This involves breaking down complex problems into smaller, more manageable steps.
If supported by the underlying model, you can surface this reasoning process to better understand how the model arrived at its final answer.

In [6]:
response = model.invoke("Why do parrots have colorful feathers?")
reasoning_steps = [b for b in response.content_blocks if b["type"] == "reasoning"]
print(" ".join(step["reasoning"] for step in reasoning_steps))

AttributeError: 'AIMessage' object has no attribute 'content_blocks'

Depending on the model, you can sometimes specify the level of effort it should put into reasoning. Similarly, you can request that the model turn off reasoning entirely. This may take the form of categorical “tiers” of reasoning (e.g., 'low' or 'high') or integer token budgets.

# Prompt Caching 

Read the doc further

# Server-side tool use

Some providers support server-side tool-calling loops: models can interact with web search, code interpreters, and other tools and analyze the results in a single conversational turn.


If a model invokes a tool server-side, the content of the response message will include content representing the invocation and result of the tool. Accessing the content blocks of the response will return the server-side tool calls and results in a provider-agnostic format:

meaning that : <b>there are no associated ToolMessage objects that need to be passed in as in client-side tool-calling</b>.

# Rate Limit

can set rate_limiter to prevent rate limit error

In [2]:
from langchain_core.rate_limiters import InMemoryRateLimiter

rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.1,  # 1 request every 10s
    check_every_n_seconds=0.1,  # Check every 100ms whether allowed to make a request
    max_bucket_size=10,  # Controls the maximum burst size.
)

model = init_chat_model(
    model="gpt-5",
    model_provider="openai",
    rate_limiter=rate_limiter  
)

In [3]:
model.invoke("Hello there!")

AIMessage(content='Hi! How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 9, 'total_tokens': 27, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Cfbw3l1ocprCj8MYHU7XHPJD5UEcO', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--187d4e44-7533-456e-945a-4dce89c092ee-0', usage_metadata={'input_tokens': 9, 'output_tokens': 18, 'total_tokens': 27, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

# Base URL

You can config base URL of other AI Brand (must be compatible with OpenAPI spec) 

In [4]:
model = init_chat_model(
    model="MODEL_NAME",
    model_provider="openai",
    base_url="BASE_URL", # <<<< e.g. Togeter AI, vLLM 
    api_key="YOUR_API_KEY",
)

# Log Probabilities

Certain models can be configured to return <b>token-level log probabilities representing the likelihood of a given token</b> by setting the logprobs parameter when initializing the model:

In [ ]:
model = init_chat_model(
    model="gpt-4o",
    model_provider="openai"
).bind(logprobs=True) 

response = model.invoke("Why do parrots talk?")
print(response.response_metadata["logprobs"])

{'content': [{'token': 'Par', 'bytes': [80, 97, 114], 'logprob': -6.704273118884885e-07, 'top_logprobs': []}, {'token': 'rots', 'bytes': [114, 111, 116, 115], 'logprob': -1.0280383548888494e-06, 'top_logprobs': []}, {'token': ' are', 'bytes': [32, 97, 114, 101], 'logprob': -0.08018330484628677, 'top_logprobs': []}, {'token': ' known', 'bytes': [32, 107, 110, 111, 119, 110], 'logprob': -0.04229895770549774, 'top_logprobs': []}, {'token': ' for', 'bytes': [32, 102, 111, 114], 'logprob': -0.0010328111238777637, 'top_logprobs': []}, {'token': ' their', 'bytes': [32, 116, 104, 101, 105, 114], 'logprob': -7.896309739408025e-07, 'top_logprobs': []}, {'token': ' ability', 'bytes': [32, 97, 98, 105, 108, 105, 116, 121], 'logprob': -0.007943740114569664, 'top_logprobs': []}, {'token': ' to', 'bytes': [32, 116, 111], 'logprob': 0.0, 'top_logprobs': []}, {'token': ' mimic', 'bytes': [32, 109, 105, 109, 105, 99], 'logprob': -0.007314950227737427, 'top_logprobs': []}, {'token': ' human', 'bytes': [3

# Token usage

A number of model providers return token usage information as part of the invocation response. When available, this information will be included on the AIMessage objects produced by the corresponding model. For more details, see the messages guide.

You can track aggregate token counts across models in an application using either a callback or context manager, as shown below:

In [ ]:
# Options 1
from langchain.chat_models import init_chat_model
from langchain_core.callbacks import UsageMetadataCallbackHandler

model_1 = init_chat_model(model="gpt-4o-mini")
model_2 = init_chat_model(model='gemini-2.5-flash', model_provider='google_genai')

callback = UsageMetadataCallbackHandler()
result_1 = model_1.invoke("Hello", config={"callbacks": [callback]})
result_2 = model_2.invoke("Hello", config={"callbacks": [callback]})
callback.usage_metadata # If there're multiple models, it will track all the tokens of all model

{'gpt-4o-mini-2024-07-18': {'input_tokens': 8,
  'output_tokens': 9,
  'total_tokens': 17,
  'input_token_details': {'audio': 0, 'cache_read': 0},
  'output_token_details': {'audio': 0, 'reasoning': 0}}}

In [ ]:
# But seem to not work with Google 
callback = UsageMetadataCallbackHandler()
result_2 = model_2.invoke("Hello", config={"callbacks": [callback]})
callback.usage_metadata # If there're multiple models, it will track all the tokens of all model

{}

In [12]:
model_2.invoke("Hello", config={"callbacks": [callback]})

AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run--963472b8-fbcb-4e51-92c1-9e9ad2f0c2a6-0', usage_metadata={'input_tokens': 2, 'output_tokens': 9, 'total_tokens': 198})

In [ ]:
# Options 2
from langchain.chat_models import init_chat_model
from langchain_core.callbacks import get_usage_metadata_callback

# model_1 = init_chat_model(model="gpt-4o-mini")
# model_2 = init_chat_model(model="claude-haiku-4-5-20251001")

with get_usage_metadata_callback() as cb:
    model_1.invoke("Hello")
    print(cb.usage_metadata)

{'gpt-4o-mini-2024-07-18': {'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}}


# Innocation config

When invoking a model, you can pass additional configuration through the config parameter using a RunnableConfig dictionary. This provides run-time control over execution behavior, callbacks, and metadata tracking.


Common configuration options include:

See doc

In [14]:
response = model.invoke(
    "Tell me a joke",
    config={
        "run_name": "joke_generation",      # Custom name for this run
        "tags": ["humor", "demo"],          # Tags for categorization
        "metadata": {"user_id": "123"},     # Custom metadata
        "callbacks": [my_callback_handler], # Callback handlers
    }
)

NameError: name 'my_callback_handler' is not defined

These configuration values are particularly useful when:

Debugging with LangSmith tracing

Implementing custom logging or monitoring

Controlling resource usage in production

Tracking invocations across complex pipelines

# Configurable models

You can actually configure the model when invoking, e.g. change model version , temperature ,blah blah

to make it differently from the default model setting


If you don’t specify a model value, then 'model' and 'model_provider' will be configurable by default.

In [16]:
from langchain.chat_models import init_chat_model

configurable_model = init_chat_model(temperature=0)

configurable_model.invoke(
    "what's your name",
    config={"configurable": {"model": "gpt-4o"}},  # Run with GPT-5-Nano
)
configurable_model.invoke(
    "what's your name",
    config={"configurable": {"model": "claude-sonnet-4-5-20250929"}},  # Run with Claude
)

AIMessage(content="I'm Claude, an AI assistant made by Anthropic.", additional_kwargs={}, response_metadata={'id': 'msg_01ASsoXdVqj1kvYXzERVYVU5', 'model': 'claude-sonnet-4-5-20250929', 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 11, 'output_tokens': 16, 'server_tool_use': None, 'service_tier': 'standard'}}, id='run--579e6931-302e-473a-814f-50db0b7eb054-0', usage_metadata={'input_tokens': 11, 'output_tokens': 16, 'total_tokens': 27, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}})